# A/B Test 사전 검증 + 간단한 성과 비교
## A/A Test + Traffic Balance Check + A/B Readout

A/B Test를 시작하기 전에 반드시 확인해야 하는 사전 검증 단계와 그 검증이 끝난 뒤 **A/B 간 차이를 간단히 읽어보는 단계**를 다룹니다.

- 사용자가 A/B 그룹에 **안정적으로 버킷팅**되는가?
- 트래픽이 의도한 비율대로 **균등하게 분배**되는가?
- 이 분배 결과를 **통계적으로 신뢰할 수 있는가?**
- 검증이 끝난 뒤, **A와 B의 기본 성과 차이를 어떻게 읽을 수 있는가?**

## 순서

1. `user_id` 기반 hash 버킷팅 구현  
2. A/A Test의 목적 이해  
3. 트래픽이 균등하게 분배되었는지 확인  
4. Proportion Z-test로 통계적으로 검증  
5. 점진적 배포(rollout) 방식의 기본 구조 이해  
6. 트래픽 검증 이후 A/B 그룹의 기본 지표 차이를 간단히 해석


## 1. 데이터 불러오기

이번 실습은 사용자 단위로 집계된 `user-level` 데이터를 사용합니다.

### 데이터 특징
- 사용자 1명 = 1행(row)
- 각 사용자별로 노출, 구매, 매출, 전환 여부가 정리되어 있음
- 실험 분석에 필요한 최소 단위 데이터를 이미 집계해둔 형태

### 핵심 컬럼
- `user_id`: 사용자 식별자
- `impression`: 노출 횟수
- `purchase`: 구매 횟수
- `revenue`: 총 매출
- `converted`: 전환 여부 (`True/False`)

### 참고
`converted`의 평균은 전환율(Conversion Rate)로 해석할 수 있습니다.  
처음에는 **트래픽 분배 검증**에 집중하고, 그 다음 단계에서 **A/B 그룹 간 기본 성과 차이**도 간단히 확인합니다.


In [4]:
import pandas as pd

df = pd.read_csv("user_agg_sample_1250_users.csv")

print("row count:", len(df))
print("columns:", df.columns.tolist())
df.head()

row count: 1250
columns: ['user_id', 'impression', 'purchase', 'revenue', 'converted']


,user_id,impression,purchase,revenue,converted
0,12346,1,74215,77183.60,True
1,12348,4,2341,1797.24,True
2,12352,8,536,2506.04,True
3,12359,4,1622,6372.58,True
4,12360,3,1165,2662.06,True


현재 데이터는 이미 사용자 단위로 집계된 상태
이제 이 데이터를 이용해 **사용자를 A/B 그룹으로 나누고**, 그 분배가 정상적인지 확인

## 2. DuckDB 환경 준비

로그 데이터나 사용자 집계 데이터를 Python만이 아니라 SQL로도 자주 확인합니다.  
그래서 같은 작업을 **Python과 DuckDB SQL 두 방식으로 모두 확인**합니다.


In [5]:
import duckdb

con = duckdb.connect("ab_test1.duckdb")

con.execute("CREATE SCHEMA IF NOT EXISTS raw_data;")
con.execute("DROP TABLE IF EXISTS raw_data.user_agg_example;")

con.execute("""
CREATE TABLE raw_data.user_agg_example (
    user_id INTEGER,
    impression INTEGER,
    purchase INTEGER,
    revenue DOUBLE,
    converted BOOLEAN
);
""")

con.execute("""
INSERT INTO raw_data.user_agg_example
SELECT *
FROM read_csv_auto('user_agg_sample_1250_users.csv');
""")

## 3. SQL 기반 버킷팅 구현

A/B Test에서는 **같은 사용자가 항상 같은 그룹에 들어가야 합니다.**

이를 위해 사용자 ID에 hash를 적용하고, 그 결과를 2로 나눈 나머지로 variant를 결정할 수 있습니다.

- `0` → A 그룹(Control)
- `1` → B 그룹(Test)

이 방식의 핵심은 **재현 가능한 랜덤(random but consistent)** 입니다.


In [6]:
con.execute("""
SELECT
    user_id,
    MOD(HASH(user_id), 2) AS variant
FROM raw_data.user_agg_example
LIMIT 10
""").df()

,user_id,variant
0,12346,0
1,12348,0
2,12352,0
3,12359,0
4,12360,1
5,12362,0
6,12372,0
7,12378,1
8,12379,0
9,12383,0


- `HASH(user_id)`: 사용자 ID를 해시값으로 변환
- `MOD(..., 2)`: 2로 나눈 나머지로 그룹 분리

같은 `user_id`는 항상 같은 hash 값을 만들기 때문에 같은 사용자는 언제 들어와도 동일한 그룹에 속합니다.


## 4. 그룹 분배 확인 (A/A Test 관점)

이제 실제로 A 그룹과 B 그룹의 사용자 수가 비슷하게 나뉘는지 확인합니다.

여기서 중요한 점은, **지금은 A와 B에 서로 다른 기능을 주지 않았다는 것**입니다.

즉, 이 단계는 **A/B Test가 아니라 A/A Test 관점의 검증**입니다.  
차이가 있어야 하는 실험이 아니라,  **차이가 없어야 정상인 상태**를 확인하는 과정


In [7]:
con.execute("""
SELECT
    variant,
    COUNT(DISTINCT user_id) AS users
FROM (
    SELECT
        user_id,
        MOD(HASH(user_id), 2) AS variant
    FROM raw_data.user_agg_example
)
GROUP BY variant
ORDER BY variant
""").df()

,variant,users
0,0,628
1,1,622


두 그룹의 사용자 수가 크게 차이 나지 않으면 버킷팅이 비교적 균등하게 이루어졌다고 볼 수 있습니다.


## 5. Python 기반 버킷팅 구현

이번에는 같은 로직을 Python으로 직접 구현해보겠습니다.  
실무에서는 서비스 런타임, 백엔드, 데이터 파이프라인 등에서 이와 유사한 방식으로 실험 그룹을 분배합니다.


In [8]:
import hashlib

def split_user(user_id):
    h = hashlib.md5(str(user_id).encode())
    return int(h.hexdigest(), 16) % 2

print(split_user(12346))
print(split_user(12408))

1
0


## 6. 전체 사용자에 대해 A/B 그룹 개수 확인

이제 사용자 전체에 대해 Python 방식으로도 A/B 그룹 수를 확인합니다.


In [9]:
users = con.execute("""
SELECT DISTINCT user_id
FROM raw_data.user_agg_example
""").df()

a = 0
b = 0

for uid in users["user_id"]:
    if split_user(uid) == 0:
        a += 1
    else:
        b += 1

print("A group:", a)
print("B group:", b)

A group: 636
B group: 614


여기서 A와 B의 사용자 수가 크게 차이 나지 않으면,  
버킷팅이 비교적 균등하게 이루어졌다고 해석할 수 있습니다.


## 7. 트래픽 배분 단계

A/B Test를 하기 전에 가장 먼저 확인해야 하는 것은 **트래픽이 정상적으로 분배되었는가** 입니다.

아직 A와 B에 서로 다른 기능을 주지 않았더라도, 사용자 수가 한쪽으로 심하게 치우쳐 있으면 실험 자체를 믿기 어렵습니다.

즉, 지금 단계는 실제 실험 전에 **버킷팅 시스템이 제대로 동작하는지 확인하는 A/A Test**를 수행합니다.


## 8. Rollout 비율 조정하기

처음에는 항상 50:50으로 실험하지는 않습니다.  
예를 들어 새로운 기능을 10% 사용자에게만 먼저 노출하고 싶을 수 있습니다.

이럴 때는 0/1 대신 0~99 범위로 나눈 뒤, 일부만 B 그룹으로 보내는 방식을 사용합니다.

In [10]:
def split_user_percent(user_id, b_percent=10):
    h = hashlib.md5(str(user_id).encode())
    val = int(h.hexdigest(), 16) % 100

    if val < b_percent:
        return 1
    else:
        return 0

users = pd.DataFrame({
    "user_id": df["user_id"].unique()
})

for uid in users["user_id"].head(20):
    print(uid, split_user_percent(uid, 10))

12346 1
12348 1
12352 0
12359 0
12360 0
12362 0
12372 0
12378 0
12379 0
12383 0
12388 0
12403 0
12408 0
12414 0
12426 0
12430 0
12431 1
12447 0
12448 0
12449 0


`b_percent=10`이면 대략 다음과 같이 해석할 수 있습니다.

- 약 90% → A 그룹(기존 버전)
- 약 10% → B 그룹(새 기능)

이런 방식은 **점진적 배포(Rollout)** 에서 자주 사용됩니다.


## 9. 데이터에 10% rollout 적용해보기

In [11]:
users["variant_10pct"] = users["user_id"].apply(lambda x: split_user_percent(x, 10))
users.head()

users["variant_10pct"].value_counts().sort_index()

variant_10pct
0    1126
1     124
Name: count, dtype: int64

- `0`: 기존 서비스 제공
- `1`: 새 기능 제공 대상

보통 이런 방식으로 먼저 소수 사용자에게만 노출해서 문제가 없는지 확인한 뒤 점차 비율을 늘립니다.


### 10% rollout 결과에서 그룹별 기본 지표 확인

아래 집계는 **실험 효과 검정**이 아니라 그룹별 기본 분포를 빠르게 확인하는 용도입니다.


In [12]:
df["variant_10pct"] = df["user_id"].apply(lambda x: split_user_percent(x, 10))

df.groupby("variant_10pct").agg({
    "user_id": "count",
    "impression": "mean",
    "purchase": "mean",
    "revenue": "mean",
    "converted": "mean"
})

,user_id,impression,purchase,revenue,converted
variant_10pct,,,,,
0,1126,3.936945,816.645648,1407.839067,0.793961
1,124,3.903226,1387.306452,1843.090161,0.854839


## 10. 50:50 버킷 결과 확인

이번에는 가장 기본적인 50:50 버킷팅 결과를 확인해보겠습니다.


In [13]:
df["variant_50pct"] = df["user_id"].apply(split_user)

df.groupby("variant_50pct").agg({
    "user_id": "count",
    "impression": "mean",
    "purchase": "mean",
    "revenue": "mean",
    "converted": "mean"
})

,user_id,impression,purchase,revenue,converted
variant_50pct,,,,,
0,636,3.977987,784.481132,1308.070613,0.795597
1,614,3.887622,965.210098,1599.083160,0.804560


이 단계에서는 아직 실험 효과를 넣지 않았기 때문에 두 그룹 간 지표 차이가 아주 크지 않은 것이 자연스럽습니다.

여기서 중요한 것은 **성과 비교 그 자체보다, 그룹 분배가 정상적인지 먼저 확인하는 것**입니다.


In [14]:
traffic_df = df["variant_50pct"].value_counts().sort_index().reset_index()
traffic_df.columns = ["variant", "user_count"]
traffic_df

,variant,user_count
0,0,636
1,1,614


- `0` → A 그룹
- `1` → B 그룹

A/A Test에서는 두 그룹이 큰 차이 없이 나뉘는 것이 자연스럽습니다.


## 11. 사용자 수를 변수로 저장

In [15]:
n_ctrl = (df["variant_50pct"] == 0).sum()
n_test = (df["variant_50pct"] == 1).sum()

print("A user count:", n_ctrl)
print("B user count:", n_test)

A user count: 636
B user count: 614


## 12. Z-score를 직접 계산해보기

A/A Test에서의 가설은 다음과 같습니다.

- **귀무가설(H₀)**: B 그룹 비율 = 0.5
- **대립가설(H₁)**: B 그룹 비율 ≠ 0.5

정상적인 50:50 분배라면 B 그룹 비율이 0.5와 크게 다르지 않아야 합니다.


In [16]:
import math

def compute_zscore(n_test, n_ctrl, p0=0.5):
    n = n_test + n_ctrl
    p = n_test / n
    se = math.sqrt(p0 * (1 - p0) / n)
    z_score = (p - p0) / se
    return z_score

z_score = compute_zscore(n_test, n_ctrl)
print("A & B user traffic comparison Z-score:", z_score)

A & B user traffic comparison Z-score: -0.6222539674441601


`z-score`가 `-1.96 ~ 1.96` 사이면 유의미한 차이가 없다고 보고 귀무가설을 기각하지 않습니다.

귀무가설을 기각하지 않으면 트래픽 분배가 정상이라고 볼 수 있습니다.


## 13. 실제 비율도 같이 확인해보기

In [17]:
n_total = n_ctrl + n_test
p_observed = n_test / n_total

print("Total users:", n_total)
print("Observed B ratio:", p_observed)
print("Expected B ratio:", 0.5)

Total users: 1250
Observed B ratio: 0.4912
Expected B ratio: 0.5


## 14. Proportion Z-test 수행

눈으로 비슷한 것과 통계적으로 같은 것은 다릅니다.  
그래서 `statsmodels`를 사용해서 비율 검정을 수행합니다.

여기서 검정하는 것은 **"B 그룹 비율이 0.5와 통계적으로 다르냐"** 입니다.


In [18]:
from statsmodels.stats.proportion import proportions_ztest

stat, pvalue = proportions_ztest(
    count=n_test,
    nobs=n_ctrl + n_test,
    value=0.5,
    alternative="two-sided"
)

print("Z-statistic:", stat)
print("P-value:", pvalue)

if pvalue > 0.05:
    print("귀무가설 기각 실패 → 트래픽이 50:50이라고 볼 수 있음")
else:
    print("귀무가설 기각 → 트래픽 분배에 문제가 있을 수 있음")

Z-statistic: -0.622350364534188
P-value: 0.5337115106595407
귀무가설 기각 실패 → 트래픽이 50:50이라고 볼 수 있음


### 해석 기준

- `p-value > 0.05`  
  → 우연한 차이로 볼 수 있음  
  → A/A Test 관점에서 정상

- `p-value <= 0.05`  
  → 차이가 너무 큼  
  → 버킷팅이나 실험 설정 문제를 의심할 수 있음


## 15. 트래픽 검증 이후, A/B를 간단히 확인

여기까지 했다면 이제 최소한 **실험 그룹 분배 자체는 신뢰할 수 있는 상태**라고 볼 수 있습니다.

그 다음에는 아주 간단하게라도 **A와 B의 기본 성과 차이**를 읽어볼 수 있어야 합니다.

여기서는 중요한 점이 있습니다.

- 지금 데이터는 실제로 A와 B에 서로 다른 기능 효과를 넣은 데이터가 아닐 수 있습니다.
- 따라서 아래 결과는 **"성과 차이를 확정하는 검정"** 이라기보다는
  **"A/B 그룹별 기본 지표를 읽는 연습"** 으로 이해하는 것이 좋습니다.

즉, 여기서는 **"B가 A보다 좋아 보이는가?"를 간단히 확인하는 수준**까지 진행합니다.


In [19]:
ab_summary = df.groupby("variant_50pct").agg({
    "user_id": "count",
    "impression": "mean",
    "purchase": "mean",
    "revenue": "mean",
    "converted": "mean"
}).reset_index()

ab_summary.columns = [
    "variant", "user_count", "avg_impression", "avg_purchase", "avg_revenue", "conversion_rate"
]

ab_summary

,variant,user_count,avg_impression,avg_purchase,avg_revenue,conversion_rate
0,0,636,3.977987,784.481132,1308.070613,0.795597
1,1,614,3.887622,965.210098,1599.083160,0.804560


위 결과에서 특히 중요하게 볼 값은 다음과 같습니다.

- `conversion_rate`: 전환율
- `avg_revenue`: 평균 매출
- `avg_purchase`: 평균 구매 횟수

보통 A/B 실험에서는 이런 지표를 기준으로 먼저 **B가 A보다 좋아 보이는지**를 1차적으로 확인합니다.


In [20]:
a_row = ab_summary[ab_summary["variant"] == 0].iloc[0]
b_row = ab_summary[ab_summary["variant"] == 1].iloc[0]

print("=== 기본 비교 ===")
print(f"A 전환율: {a_row['conversion_rate']:.4f}")
print(f"B 전환율: {b_row['conversion_rate']:.4f}")
print(f"A 평균 매출: {a_row['avg_revenue']:.4f}")
print(f"B 평균 매출: {b_row['avg_revenue']:.4f}")
print(f"A 평균 구매: {a_row['avg_purchase']:.4f}")
print(f"B 평균 구매: {b_row['avg_purchase']:.4f}")

=== 기본 비교 ===
A 전환율: 0.7956
B 전환율: 0.8046
A 평균 매출: 1308.0706
B 평균 매출: 1599.0832
A 평균 구매: 784.4811
B 평균 구매: 965.2101


## 16. B가 A보다 좋아 보이는지 해석

이 단계에서는 복잡한 검정보다 먼저 **방향성**을 읽는 것이 중요합니다.

- B의 전환율이 더 높다  
  → B가 전환 측면에서 더 좋아 보인다
- B의 평균 매출이 더 높다  
  → B가 매출 측면에서 더 좋아 보인다

즉, **트래픽이 정상이라는 전제 아래**에 이제야 비로소 A/B의 결과를 읽기 시작할 수 있는 것입니다.

다만 이 해석은 아직 **"눈으로 보는 1차 비교"** 에 가깝습니다.

정말로 믿을 수 있는 차이인지까지 판단하려면 그 다음 단계에서 **실제 성과 지표에 대한 통계 검정**이 추가로 필요합니다.


In [21]:
if b_row["conversion_rate"] > a_row["conversion_rate"]:
    print("전환율 기준으로는 B가 A보다 좋아 보입니다.")
elif b_row["conversion_rate"] < a_row["conversion_rate"]:
    print("전환율 기준으로는 A가 B보다 좋아 보입니다.")
else:
    print("전환율 기준으로는 A와 B가 동일합니다.")

전환율 기준으로는 B가 A보다 좋아 보입니다.


## 17. 실제 A/B 성과 검정: 전환율 차이에 대한 Proportion Z-test

여기까지는
A와 B의 결과를 눈으로 1차 확인한 단계였습니다.

하지만 실제 A/B Test에서는
“B가 더 좋아 보인다”에서 끝나면 안 되고,
그 차이가 통계적으로도 유의한지 확인해야 합니다.

이번에는 트래픽 비율이 아니라
A 그룹 전환율과 B 그룹 전환율이 실제로 다른지 검정해보겠습니다.

가설은 다음과 같습니다.

귀무가설(H₀): A와 B의 전환율은 같다
대립가설(H₁): A와 B의 전환율은 다르다

즉, 여기서부터가 진짜 A/B 성과 검정입니다.

In [22]:
from statsmodels.stats.proportion import proportions_ztest

# A/B 그룹 분리
group_A = df[df["variant_50pct"] == 0]
group_B = df[df["variant_50pct"] == 1]

# 전환 성공 수
success_A = group_A["converted"].sum()
success_B = group_B["converted"].sum()

# 전체 사용자 수
n_A = len(group_A)
n_B = len(group_B)

print("A conversions:", success_A)
print("B conversions:", success_B)
print("A users:", n_A)
print("B users:", n_B)

A conversions: 506
B conversions: 494
A users: 636
B users: 614


In [23]:
# 두 집단 비율 검정
count = [success_A, success_B]
nobs = [n_A, n_B]

stat_ab, pvalue_ab = proportions_ztest(
    count=count,
    nobs=nobs,
    alternative="two-sided"
)

print("A/B conversion Z-statistic:", stat_ab)
print("A/B conversion P-value:", pvalue_ab)

A/B conversion Z-statistic: -0.3960411410672107
A/B conversion P-value: 0.6920746781371256


In [24]:
# 전환율 계산
conv_A = success_A / n_A
conv_B = success_B / n_B

print("A conversion rate:", round(conv_A, 4))
print("B conversion rate:", round(conv_B, 4))

if pvalue_ab < 0.05:
    print("귀무가설 기각 → A와 B의 전환율 차이는 통계적으로 유의함")
    if conv_B > conv_A:
        print("결론: B가 A보다 전환율 측면에서 유의하게 더 좋다고 해석할 수 있음")
    elif conv_B < conv_A:
        print("결론: A가 B보다 전환율 측면에서 유의하게 더 좋다고 해석할 수 있음")
    else:
        print("결론: 계산상 차이는 거의 없음")
else:
    print("귀무가설 기각 실패 → 전환율 차이를 우연으로 볼 수 있음")
    if conv_B > conv_A:
        print("B가 숫자상으로는 더 높아 보이지만, 통계적으로는 확정할 수 없음")
    elif conv_B < conv_A:
        print("A가 숫자상으로는 더 높아 보이지만, 통계적으로는 확정할 수 없음")
    else:
        print("두 그룹의 전환율은 사실상 동일함")

A conversion rate: 0.7956
B conversion rate: 0.8046
귀무가설 기각 실패 → 전환율 차이를 우연으로 볼 수 있음
B가 숫자상으로는 더 높아 보이지만, 통계적으로는 확정할 수 없음


In [25]:
from scipy import stats

a = df["variant_50pct"].to_numpy()
a[:10]

stats.ttest_1samp(a, 0.5)

TtestResult(statistic=np.float64(-0.6221013745804199), pvalue=np.float64(0.5339887539624947), df=np.int64(1249))

## 18. 최종 정리

이번 실습에서 확인한 핵심은 다음과 같습니다.

1. `hash` 기반으로 사용자를 안정적으로 A/B 그룹에 나눌 수 있다.  
2. A/A Test에서는 B 그룹 비율이 0.5와 크게 다르지 않아야 한다.  
3. Z-score를 직접 계산해서 분배 이상 여부를 빠르게 확인할 수 있다.  
4. Proportion Z-test로 p-value를 구해 통계적으로 검증할 수 있다.  
5. p-value가 충분히 크면 트래픽 분배가 정상이라고 해석할 수 있다.  
6. 트래픽 검증이 끝난 뒤에는 A/B 그룹의 기본 성과를 간단히 읽어볼 수 있다.  